# TCC — Pipeline completo no Google Colab

Notebook único do projeto para GPU/Drive/GitHub, clustering/revisão e experimentos YOLO26 Detection/Segmentation em Base, LoRA e Full fine-tune.

**Importante:** este notebook não atualiza `pip`, `setuptools` ou `wheel`. Isso evita o aviso do Colab sobre `_distutils_hack` e a necessidade de reiniciar o runtime durante o setup.

Antes de executar, selecione **Runtime > Change runtime type > GPU** e configure em **Secrets**: `GITHUB_TOKEN`, `ROBOFLOW_API_KEY` e, opcionalmente, `NGROK_AUTHTOKEN`.


## 1. Verificar a GPU sem importar pacotes que serão instalados


In [ ]:
import os, sys, subprocess
print("Python:", sys.version.split()[0])
subprocess.run(["nvidia-smi"], check=False)


## 2. Montar Google Drive


In [ ]:
from google.colab import drive, userdata
from pathlib import Path

drive.mount("/content/drive")
RUNTIME = Path("/content/drive/MyDrive/TCC_Code_Test_runtime")
for name in ["data", "workspace", "exports", "weights"]:
    (RUNTIME / name).mkdir(parents=True, exist_ok=True)
print("Persistência:", RUNTIME)


## 3. Clonar ou atualizar o repositório privado


In [ ]:
REPO = Path("/content/TCC_Code_Test")
token = userdata.get("GITHUB_TOKEN")
if not token:
    raise RuntimeError("GITHUB_TOKEN não encontrado em Colab > Secrets.")

netrc = Path.home() / ".netrc"
netrc.write_text(f"machine github.com\nlogin x-access-token\npassword {token}\n", encoding="utf-8")
netrc.chmod(0o600)
try:
    if REPO.exists():
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "https://github.com/ferreis/TCC_Code_Test.git", str(REPO)], check=True)
finally:
    netrc.unlink(missing_ok=True)
os.chdir(REPO)
print("Commit TCC:", subprocess.check_output(["git","rev-parse","--short","HEAD"], text=True).strip())


## 4. Instalar dependências do Colab

Não execute `pip install --upgrade pip setuptools wheel` neste runtime. O Colab já traz essas ferramentas e atualizá-las após o kernel iniciar pode disparar o aviso `_distutils_hack`.


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-r", "requirements-colab.txt"],
    check=True,
)
print("Dependências do projeto instaladas.")


## 5. Instalar o backend YOLO-Master antes do primeiro `import ultralytics`

O fork é instalado agora, antes de qualquer import do pacote `ultralytics`. Assim Base, LoRA e Full usam a mesma implementação nesta sessão e não é necessário reiniciar o runtime para trocar de backend.


In [ ]:
subprocess.run(
    [sys.executable, "scripts/setup_yolo_master.py", "--dir", "/content/YOLO-Master"],
    check=True,
)
print("YOLO-Master instalado.")


## 6. Validar PyTorch, CUDA, Ultralytics e LoRA


In [ ]:
import importlib.util
import torch
import ultralytics

print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU:", p.name)
    print("VRAM:", round(p.total_memory / 1024**3, 2), "GB")
    print("Compute capability:", f"{p.major}.{p.minor}")
print("Ultralytics/YOLO-Master:", ultralytics.__version__)
print("Caminho:", ultralytics.__file__)
lora_ok = importlib.util.find_spec("ultralytics.utils.lora") is not None
print("LoRA disponível:", lora_ok)
if not lora_ok:
    raise RuntimeError("O backend LoRA não foi carregado. Verifique a saída da etapa 5 antes de continuar.")


## 7. Conectar data/workspace/exports/weights ao Drive


In [ ]:
import shutil
for name in ["data", "workspace", "exports", "weights"]:
    target = RUNTIME / name
    link = REPO / name
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        if link.is_dir():
            for p in link.iterdir():
                dst = target / p.name
                if not dst.exists():
                    shutil.copytree(p, dst) if p.is_dir() else shutil.copy2(p, dst)
            shutil.rmtree(link)
        else:
            link.unlink()
    link.symlink_to(target, target_is_directory=True)
print("Drive conectado ao projeto.")


## 8. Configurar Roboflow e baixar dataset


In [ ]:
rf_key = userdata.get("ROBOFLOW_API_KEY")
if not rf_key:
    raise RuntimeError("ROBOFLOW_API_KEY não encontrado em Colab > Secrets.")
os.environ["ROBOFLOW_API_KEY"] = rf_key
os.environ["ROBOFLOW_WORKSPACE"] = "rafael-fernando-dos-reis-mecabo"
os.environ["ROBOFLOW_PROJECT"] = "detecao-de-cetaceos-afrzs-iqey9"
os.environ["ROBOFLOW_VERSION"] = "1"

images_dir = Path("data/roboflow/train/images")
if images_dir.exists() and any(images_dir.iterdir()):
    print("Dataset já existe:", images_dir)
else:
    subprocess.run([
        sys.executable, "scripts/download_roboflow.py", "--format", "yolo26",
        "--output", "data/roboflow", "--overwrite",
    ], check=True)


## 9. Baixar SAM


In [ ]:
sam = Path("weights/sam_vit_b_01ec64.pth")
if sam.exists():
    print("SAM já existe:", sam)
else:
    subprocess.run([sys.executable, "scripts/download_sam.py"], check=True)


## 10. Clustering das bounding boxes já revisadas — opcional


In [ ]:
from tcc_discovery.box_clustering import BoxClusteringConfig, run_box_clustering
project_dir = Path("workspace/bbox_clusters_colab")
if (project_dir / "catalog.json").exists():
    print("Projeto já existe:", project_dir)
else:
    cfg = BoxClusteringConfig(
        context_scale=1.8, min_crop_size_px=64, min_cluster_size=6,
        embedding_batch_size=8, dino_cpu_fallback_on_oom=True,
    )
    result = run_box_clustering(
        "data/roboflow/train", project_dir, cfg, overwrite=False,
        progress_cb=lambda d,t,m: print(f"[{d}/{t}] {m}") if d % 100 == 0 or d == t else None,
    )
    print("Imagens:", len(result["images"]), "Objetos:", len(result["objects"]))


## 11. Abrir Streamlit — opcional

Use para upload de imagem/vídeo, classificação de clusters e revisão. Requer `NGROK_AUTHTOKEN`.


In [ ]:
from pyngrok import ngrok
import time
ngrok_token = userdata.get("NGROK_AUTHTOKEN")
if not ngrok_token:
    print("NGROK_AUTHTOKEN não configurado; etapa opcional pulada.")
else:
    ngrok.kill()
    ngrok.set_auth_token(ngrok_token)
    env = os.environ.copy()
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    streamlit_process = subprocess.Popen(
        ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
        env=env, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )
    time.sleep(3)
    public_url = ngrok.connect(8501, bind_tls=True).public_url
    print("Abra a interface:", public_url)
    print("Streamlit PID:", streamlit_process.pid)


## 12. Encerrar Streamlit antes dos treinamentos


In [ ]:
if "streamlit_process" in globals() and streamlit_process.poll() is None:
    streamlit_process.terminate()
    print("Streamlit encerrado.")
try:
    ngrok.kill()
except Exception:
    pass
if torch.cuda.is_available():
    torch.cuda.empty_cache()
subprocess.run(["nvidia-smi"], check=False)


# Treinamento YOLO26

Matriz inicial com YOLO26n:
- Detection: Base, LoRA e Full fine-tune.
- Segmentation: Base, LoRA e Full fine-tune.


## 13. Validar a matriz experimental


In [ ]:
subprocess.run([sys.executable, "scripts/train_experiments.py", "--group", "all", "--list"], check=True)
subprocess.run([sys.executable, "scripts/train_experiments.py", "--group", "experiments", "--dry-run"], check=True)


## 14. Smoke test YOLO26n Detection — 3 épocas


In [ ]:
subprocess.run([
    sys.executable, "scripts/train_experiments.py", "--group", "smoke_tests",
    "--only", "detect_yolo26n_smoke",
], check=True)


## 15. Detection: Base + LoRA + Full fine-tune


In [ ]:
subprocess.run([
    sys.executable, "scripts/train_experiments.py",
    "--only", "detect_base", "detect_lora", "detect_full",
], check=True)


## 16. Segmentation: smoke test + Base + LoRA + Full


In [ ]:
seg = Path("data/segmentation/data.yaml")
print("Dataset de segmentação pronto:", seg.exists(), "-", seg)
if seg.exists():
    subprocess.run([
        sys.executable, "scripts/train_experiments.py", "--group", "smoke_tests",
        "--only", "segment_yolo26n_smoke",
    ], check=True)
    subprocess.run([
        sys.executable, "scripts/train_experiments.py",
        "--only", "segment_base", "segment_lora", "segment_full",
    ], check=True)
else:
    print("Segmentação ainda não está pronta; etapa pulada.")


## 17. Comparar parâmetros e métricas


In [ ]:
import pandas as pd
summary = Path("workspace/training_runs/experiment_summary.csv")
if summary.exists():
    df = pd.read_csv(summary)
    display(df)
else:
    print("A tabela aparecerá após o primeiro experimento.")


## 18. Retomar treinamento após desconexão


In [ ]:
# Descomente somente o experimento necessário:
# subprocess.run([sys.executable, "scripts/train_experiments.py", "--only", "detect_base", "--resume"], check=True)
# subprocess.run([sys.executable, "scripts/train_experiments.py", "--only", "detect_lora", "--resume"], check=True)
# subprocess.run([sys.executable, "scripts/train_experiments.py", "--only", "detect_full", "--resume"], check=True)
# subprocess.run([sys.executable, "scripts/train_experiments.py", "--only", "segment_base", "--resume"], check=True)
# subprocess.run([sys.executable, "scripts/train_experiments.py", "--only", "segment_lora", "--resume"], check=True)
# subprocess.run([sys.executable, "scripts/train_experiments.py", "--only", "segment_full", "--resume"], check=True)


## Resultado persistente

Datasets, catálogos, checkpoints e métricas ficam em `MyDrive/TCC_Code_Test_runtime/`. Os pacotes Python em `/content` precisam ser reinstalados a cada novo runtime, mas o notebook faz isso nas etapas 4 e 5.
